# 6. Distractores duros: ¿identifica el gráfico o solo el tema?

**Proyecto 2 MCC225** · Tercer control de la serie

## La objeción que este notebook responde

Los controles anteriores dejaron el R@1 de CLIP-L en 0.326 sin título. Alguien
puede objetar, con razón: *"0.33 sigue siendo diez veces el azar; algo está
leyendo"*.

Es una objeción legítima y hay que responderla con datos. La hipótesis
alternativa es que ese 0.33 se explique por **reconocimiento temático**: basta
distinguir un gráfico de morosidad de uno de crecimiento mundial, sin leer nada
del contenido. Los 26 a 40 candidatos de cada pool cubren temas muy distintos,
así que la tarea puede resolverse por tema.

Este control lo pone a prueba restringiendo el pool a gráficos del **mismo
capítulo** del informe: comparten tema, vocabulario técnico y estilo visual. Si
el desempeño se desploma, lo que quedaba era reconocimiento temático.

## El control de tamaño, que es lo que hace válida la comparación

Comparar el R@1 del pool completo (40 candidatos) contra el del pool duro (6
candidatos) **no sería válido**: con menos candidatos el R@1 sube por
construcción, sin que el modelo mejore.

Por eso cada pool duro se compara contra un **pool aleatorio del mismo tamaño**,
formado con gráficos de otros capítulos. Ambos tienen los mismos candidatos; solo
cambia si son del mismo tema o no.

| Comparación | Qué aísla |
|---|---|
| duro vs. aleatorio, mismo tamaño | dificultad por **similitud temática** |
| pool completo vs. pool reducido | dificultad por **número de candidatos** |

La primera es la que responde la pregunta. La segunda se reporta aparte para que
no se confundan.

In [1]:
from pathlib import Path
import json, re, sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MULTI = BASE / "multiedicion"

# --- variante de imágenes -------------------------------------------------
# Un único punto decide si se trabaja con las imágenes que llevan el título
# impreso o con las recortadas. El manifiesto es el mismo para ambas; cambia
# solo la subcarpeta. El nombre del directorio de salida arrastra la variante,
# de modo que dos corridas nunca se sobrescriben.
import sys
sys.path.insert(0, str(BASE))
from config_variante import (VARIANTE, carpeta_imagenes, ruta_imagen,
                             dir_salida, resumen)

IMGS = carpeta_imagenes(MULTI)
print(resumen(MULTI))

OUT = dir_salida(BASE, "resultados_distractores")

SEED = 22514
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_SPECS = {
    "CLIP":     {"checkpoint": "openai/clip-vit-base-patch32", "max_tokens": 77},
    "CLIP-L":   {"checkpoint": "openai/clip-vit-large-patch14", "max_tokens": 77},
    "LongCLIP": {"checkpoint": "zer0int/LongCLIP-GmP-ViT-L-14", "max_tokens": 248},
}
CAPTIONS = ["caption_2", "caption_3"]
N_REPETICIONES = 20           # pools aleatorios por ítem, para promediar el azar

man = pd.read_csv(MULTI / "manifest_multiedicion.csv").reset_index(drop=True)
print(f"{len(man)} pares | dispositivo: {DEVICE}")

variante: con_titulo | carpeta: con_titulo | 95 imágenes
95 pares | dispositivo: cuda


## 6.1 Agrupación por capítulo

El identificador del gráfico codifica su ubicación en el informe: `III.A.2`
pertenece al capítulo III.A, `V.6` al capítulo V. Esa estructura ya agrupa por
tema sin necesidad de etiquetar nada a mano.

Se usa el nivel de **sección** (el número romano) porque el de capítulo deja
demasiados grupos de uno o dos elementos: un pool de un solo candidato daría R@1
= 1 siempre y no mediría nada.

In [2]:
def seccion(chart_id):
    m = re.match(r"([IVX]+)", str(chart_id))
    return m.group(1) if m else "?"

def capitulo(chart_id):
    m = re.match(r"([IVX]+)(?:\.([A-Z]))?\.", str(chart_id))
    if not m:
        return "?"
    return f"{m.group(1)}.{m.group(2)}" if m.group(2) else m.group(1)

man["seccion"] = man["chart_id"].map(seccion)
man["capitulo"] = man["chart_id"].map(capitulo)

MIN_GRUPO = 4      # el pool duro debe tener al menos 4 candidatos

tam = man.groupby(["edicion", "seccion"]).size()
print(tam.to_string())

validos = [i for i, r in man.iterrows()
           if tam.get((r["edicion"], r["seccion"]), 0) >= MIN_GRUPO]
print(f"\nítems evaluables (grupo >= {MIN_GRUPO}): {len(validos)} de {len(man)}")

edicion  seccion
2021-1   I          14
         II          4
         III         2
         IV          1
         V           5
2024-2   I          18
         II          3
         III         7
         IV          1
2026-1   I           2
         II          6
         III         7
         IV         12
         V           6
         VI          4
         VII         3

ítems evaluables (grupo >= 4): 83 de 95


## 6.2 Embeddings

Se calculan una sola vez por modelo y caption; los pools se arman después
indexando la misma matriz. Así el pool duro y el aleatorio comparten exactamente
los mismos vectores y la comparación no puede contaminarse.

In [3]:
from transformers import CLIPModel, CLIPProcessor, CLIPConfig

def cargar_modelo(nombre):
    spec = MODEL_SPECS[nombre]
    ck, maxlen = spec["checkpoint"], spec["max_tokens"]
    if maxlen > 77:
        cfg = CLIPConfig.from_pretrained(ck)
        cfg.text_config.max_position_embeddings = maxlen
        modelo = CLIPModel.from_pretrained(ck, config=cfg, use_safetensors=True)
    else:
        modelo = CLIPModel.from_pretrained(ck, use_safetensors=True)
    proc = CLIPProcessor.from_pretrained(ck)
    modelo = modelo.to(DEVICE).eval()
    assert modelo.config.text_config.max_position_embeddings == maxlen
    return modelo, proc


@torch.no_grad()
def embeddings(modelo, proc, rutas, textos, maxlen, batch=8):
    im = []
    for s in range(0, len(rutas), batch):
        imgs = [Image.open(p).convert("RGB") for p in rutas[s:s + batch]]
        e = modelo.get_image_features(**proc(images=imgs, return_tensors="pt").to(DEVICE))
        im.append(F.normalize(e, dim=-1).cpu())
    tx = []
    for s in range(0, len(textos), batch):
        inp = proc(text=list(textos[s:s + batch]), return_tensors="pt",
                   padding=True, truncation=True, max_length=maxlen).to(DEVICE)
        e = modelo.get_text_features(**inp)
        tx.append(F.normalize(e, dim=-1).cpu())
    return torch.cat(im).numpy(), torch.cat(tx).numpy()


rutas = [ruta_imagen(MULTI, p) for p in man["image_path"]]
EMB = {}
for nombre in MODEL_SPECS:
    modelo, proc = cargar_modelo(nombre)
    for cap in CAPTIONS:
        Ei, Et = embeddings(modelo, proc, rutas, man[cap].astype(str).tolist(),
                            MODEL_SPECS[nombre]["max_tokens"])
        EMB[(nombre, cap)] = (Ei, Et)
        print(f"  {nombre} · {cap}: {Ei.shape}")
    del modelo
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

/tf/work/torch_gpu_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


  CLIP · caption_2: (95, 512)
  CLIP · caption_3: (95, 512)
  CLIP-L · caption_2: (95, 768)
  CLIP-L · caption_3: (95, 768)
  LongCLIP · caption_2: (95, 768)
  LongCLIP · caption_3: (95, 768)


## 6.3 Evaluación por pool

Para cada ítem se construyen dos pools del mismo tamaño y se mide en cuál queda
el caption correcto en primer lugar.

Los pools aleatorios se repiten 20 veces por ítem: un solo sorteo tendría
demasiada varianza y podría dar la diferencia por casualidad.

In [4]:
def rango_en_pool(Ei, Et, i, candidatos):
    """Posición del caption correcto cuando la imagen i compite contra
    'candidatos' (que incluye a i)."""
    sims = Ei[i] @ Et[candidatos].T
    orden = np.argsort(-sims)
    pos_correcta = list(candidatos).index(i)
    return int(np.where(orden == pos_correcta)[0][0]) + 1


filas = []
for (nombre, cap), (Ei, Et) in EMB.items():
    for i in validos:
        r = man.iloc[i]
        # mismos candidatos de edición, para no mezclar pools
        misma_ed = man.index[man["edicion"] == r["edicion"]].to_numpy()

        duros = np.array([j for j in misma_ed
                          if man.iloc[j]["seccion"] == r["seccion"] and j != i])
        otros = np.array([j for j in misma_ed
                          if man.iloc[j]["seccion"] != r["seccion"]])

        # El pool aleatorio debe tener el MISMO tamaño que el duro. En las
        # secciones grandes no hay suficientes gráficos de otras secciones para
        # igualarlo, así que se recorta el pool duro en vez de descartar el
        # ítem: perder los capítulos más numerosos sesgaría la muestra hacia
        # los pools pequeños, que son justamente los más fáciles.
        k = min(len(duros), len(otros)) + 1
        if k < MIN_GRUPO:
            continue
        sel_duros = rng.choice(duros, size=k - 1, replace=False)
        rk_duro = rango_en_pool(Ei, Et, i, np.concatenate([[i], sel_duros]))

        # pools aleatorios del mismo tamaño, con gráficos de otras secciones
        rk_azar = []
        for _ in range(N_REPETICIONES):
            sel = rng.choice(otros, size=k - 1, replace=False)
            rk_azar.append(rango_en_pool(Ei, Et, i, np.concatenate([[i], sel])))

        filas.append({
            "modelo": nombre, "caption": cap, "edicion": r["edicion"],
            "chart_id": r["chart_id"], "seccion": r["seccion"], "k": k,
            "rango_duro": rk_duro,
            "acierto_duro": int(rk_duro == 1),
            "rango_azar": float(np.mean(rk_azar)),
            "acierto_azar": float(np.mean([x == 1 for x in rk_azar])),
        })

det = pd.DataFrame(filas)
det.to_csv(OUT / "detalle_por_item.csv", index=False)
print(f"{len(det)} evaluaciones | tamaño de pool: {det.k.min()}–{det.k.max()} "
      f"(mediana {int(det.k.median())})")

498 evaluaciones | tamaño de pool: 4–13 (mediana 12)


## 6.4 Resultado

La columna que importa es la última: cuánto cae el R@1 al reemplazar candidatos
aleatorios por gráficos del mismo capítulo, **manteniendo el número de
candidatos**.

In [5]:
res = (det.groupby(["modelo", "caption"])
          .agg(n=("acierto_duro", "size"),
               R1_pool_duro=("acierto_duro", "mean"),
               R1_pool_aleatorio=("acierto_azar", "mean"))
          .round(4).reset_index())
res["caida"] = (res["R1_pool_duro"] - res["R1_pool_aleatorio"]).round(4)
res["caida_relativa_%"] = (100 * res["caida"] / res["R1_pool_aleatorio"]).round(1)
res.to_csv(OUT / "resumen_distractores.csv", index=False)
print(res.to_string(index=False))

  modelo   caption  n  R1_pool_duro  R1_pool_aleatorio   caida  caida_relativa_%
    CLIP caption_2 83        0.5181             0.5982 -0.0801             -13.4
    CLIP caption_3 83        0.6265             0.7090 -0.0825             -11.6
  CLIP-L caption_2 83        0.7229             0.8030 -0.0801             -10.0
  CLIP-L caption_3 83        0.7108             0.8765 -0.1657             -18.9
LongCLIP caption_2 83        0.6506             0.7283 -0.0777             -10.7
LongCLIP caption_3 83        0.7108             0.8578 -0.1470             -17.1


### Prueba p_mcnemar

Cada ítem se evalúa en ambos pools, así que la comparación es pareada. Se cuentan
los casos en que el modelo acierta en el pool aleatorio y falla en el duro, frente
a los inversos.

In [6]:
from math import comb

def p_mcnemar(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)


filas = []
for (nombre, cap), g in det.groupby(["modelo", "caption"]):
    # el pool aleatorio da una tasa; se compara contra el resultado binario duro
    solo_azar = int(((g.acierto_azar > 0.5) & (g.acierto_duro == 0)).sum())
    solo_duro = int(((g.acierto_azar <= 0.5) & (g.acierto_duro == 1)).sum())
    p = p_mcnemar(solo_azar, solo_duro)
    filas.append({"modelo": nombre, "caption": cap, "n": len(g),
                  "solo_aleatorio": solo_azar, "solo_duro": solo_duro,
                  "p_valor": round(p, 4), "significativo_0.05": p < 0.05})

mc = pd.DataFrame(filas)
mc.to_csv(OUT / "mcnemar_distractores.csv", index=False)
print(mc.to_string(index=False))

  modelo   caption  n  solo_aleatorio  solo_duro  p_valor  significativo_0.05
    CLIP caption_2 83              13          5   0.0963               False
    CLIP caption_3 83              15          8   0.2100               False
  CLIP-L caption_2 83               8          1   0.0391                True
  CLIP-L caption_3 83              15          1   0.0005                True
LongCLIP caption_2 83               8          2   0.1094               False
LongCLIP caption_3 83              14          1   0.0010                True


## 6.5 Qué secciones son más difíciles

Desagregación exploratoria. Con pocos ítems por sección esto orienta dónde mirar,
no permite comparar secciones entre sí.

In [7]:
por_sec = (det[det.modelo == "CLIP-L"]
             .groupby(["edicion", "seccion"])
             .agg(n=("acierto_duro", "size"), k=("k", "first"),
                  R1_duro=("acierto_duro", "mean"),
                  R1_azar=("acierto_azar", "mean"))
             .round(3))
por_sec["caida"] = (por_sec.R1_duro - por_sec.R1_azar).round(3)
por_sec.to_csv(OUT / "por_seccion.csv")
print("CLIP-L, promediado sobre los captions evaluados:")
print(por_sec.sort_values("caida").to_string())

CLIP-L, promediado sobre los captions evaluados:
                  n   k  R1_duro  R1_azar  caida
edicion seccion                                 
2026-1  IV       24  12    0.625    0.840 -0.215
2021-1  I        28  13    0.679    0.857 -0.178
2026-1  III      14   7    0.571    0.739 -0.168
2024-2  I        36  12    0.722    0.889 -0.167
2026-1  II       12   6    0.750    0.867 -0.117
2021-1  V        10   5    0.600    0.695 -0.095
2024-2  III      14   7    0.643    0.693 -0.050
2021-1  II        8   4    1.000    1.000  0.000
2026-1  VI        8   4    0.875    0.856  0.019
        V        12   6    1.000    0.917  0.083


## 6.6 Cómo leer el resultado

| Patrón | Interpretación |
|---|---|
| Caída grande (>0.20) | El desempeño dependía de distinguir temas. Frente a gráficos del mismo capítulo el modelo pierde capacidad de identificación, lo que confirma que no está leyendo contenido específico |
| Caída moderada (0.05–0.20) | Hay identificación específica, pero la similitud temática la degrada apreciablemente |
| Sin caída | El modelo identifica el gráfico por su contenido, no por su tema. Sería el resultado más favorable y obligaría a revisar la interpretación de los notebooks anteriores |

**Cierre de la serie de controles.** Con este notebook quedan medidos los tres
atajos que el modelo podría estar usando en lugar de leer el gráfico:

| Control | Atajo eliminado | Notebook |
|---|---|---|
| Ablación del título | leer texto impreso | 3 (variante sin título) |
| Prueba composicional | reconocer vocabulario | 4 |
| Distractores duros | distinguir temas | 6 (este) |

**Limitaciones.** Los pools duros son pequeños (4 a 18 candidatos) y su tamaño
varía entre secciones, de modo que el promedio mezcla situaciones distintas; por
eso se reporta también la desagregación. El agrupamiento por sección es una
aproximación al tema: dos gráficos de la misma sección pueden ser visualmente
muy distintos, lo que hace de este un control **conservador** —subestima la
dificultad real de un pool temáticamente homogéneo.

In [8]:
config = {
    "experimento": "distractores_duros",
    "pregunta": "el R@1 restante, ¿es identificación específica o reconocimiento temático?",
    "agrupacion": "sección del informe (número romano del chart_id)",
    "min_grupo": MIN_GRUPO,
    "control_de_tamano": "pool aleatorio del mismo tamaño, otras secciones",
    "repeticiones_aleatorias": N_REPETICIONES,
    "variante_imagenes": VARIANTE,
    "captions": CAPTIONS,
    "seed": SEED,
    "n_items_evaluables": len(validos),
    "n_total": int(len(man)),
    "runtime": {"python": sys.version.split()[0], "torch": torch.__version__,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
}
with open(OUT / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(json.dumps(config, ensure_ascii=False, indent=2))
print(f"\nsalidas en {OUT}")

{
  "experimento": "distractores_duros",
  "pregunta": "el R@1 restante, ¿es identificación específica o reconocimiento temático?",
  "agrupacion": "sección del informe (número romano del chart_id)",
  "min_grupo": 4,
  "control_de_tamano": "pool aleatorio del mismo tamaño, otras secciones",
  "repeticiones_aleatorias": 20,
  "variante_imagenes": "con_titulo",
  "captions": [
    "caption_2",
    "caption_3"
  ],
  "seed": 22514,
  "n_items_evaluables": 83,
  "n_total": 95,
  "runtime": {
    "python": "3.11.0rc1",
    "torch": "2.5.1+cu121",
    "gpu": "NVIDIA GeForce RTX 4070 Laptop GPU"
  }
}

salidas en /tf/work/final/sbs_iesf_pares/resultados_distractores_con_titulo
